## Submit pipelines, deploy multiple models on AI-Platform (Multi-Model Triton Endpoint)

#### Notebook performs full End-to-End scenario:
1. Train a **3-stage Iris classification pipeline** (sklearn RandomForestClassifier) on AML
2. Train a **PyTorch sine-wave MLP** locally
3. Package both models into a single **Triton model repository**
4. Deploy to a single **AKS Kubernetes Online Endpoint** with **distinct per-model URLs**
5. Test inference on both models via the KFServing V2 protocol

**Routing mechanisms** (tried in order):
- Triton-native path: `POST {scoring_uri}/v2/models/{model_name}/infer`
- Query parameter:    `POST {scoring_uri}?model={model_name}`
- Body field fallback: `{"model_name": "...", "inputs": [...]}`

## Install Required Packages

Install all dependencies including **PyTorch** for the second model. Run once per kernel restart.

In [1]:
%pip install "azure-ai-ml>=1.12.0" \
             "azure-identity>=1.14.0" \
             "scikit-learn>=1.3.0" \
             "skl2onnx>=1.16.0" \
             "onnx>=1.14.0" \
             "onnxruntime>=1.16.0" \
             "numpy>=1.24.0" \
             "requests>=2.31.0" \
             "pandas>=2.0.0" \
             "torch>=2.0.0"
# skl2onnx/onnx/onnxruntime are required to convert the sklearn model to ONNX format
# for the Triton ONNX Runtime backend.
# NOTE: The standard nvcr/nvidia/tritonserver:23.08-py3 image ships the ONNX Runtime
# backend but NOT the FIL (Forest Inference Library) backend — FIL requires a
# separate RAPIDS container build. We therefore use the ONNX Runtime backend here.

# torch is required to train the PyTorch sine-wave MLP and export it to ONNX.

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ── Project Root Setup ────────────────────────────────────────────────────────
# Ensure the working directory is always the project root so that all
# relative paths (./src/..., ./artifacts/...) resolve correctly regardless
# of where the Jupyter server was launched from.
#
# If you opened Jupyter from outside this folder, update the path below:
#   os.chdir("/absolute/path/to/ai-platform-aml-triton-examples")
import os, sys
from pathlib import Path

# Detect the directory that contains this notebook file (works in VS Code + JupyterLab)
_nb_file = globals().get('__vsc_ipynb_file__') or globals().get('__file__', None)
if _nb_file:
    _project_root = str(Path(_nb_file).resolve().parent)
    os.chdir(_project_root)
else:
    _project_root = os.getcwd()

# Make src.* importable
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

# Create artifacts directory for runtime outputs (gitignored)
os.makedirs(os.path.join(_project_root, 'artifacts'), exist_ok=True)

print(f'Project root : {_project_root}')
print(f'Working dir  : {os.getcwd()}')


Project root : /home/jovyan/ai-platform-aml-triton-examples
Working dir  : /home/jovyan/ai-platform-aml-triton-examples


| Compute Name | Time Slicing Options     | Base Node Guaranteed CPU | Base Node Guaranteed Memory |
|--------------|------------------|---------------|------------------|
| cpu          | -2, -4           | 14            | 46Gi             |
| gput41       | -2, -4           | 6             | 44Gi             |
| gpuv1001     | -2, -4           | 4             | 98Gi             |
| gpuv1002     | -2, -4           | 10            | 206Gi            |
| gpua100      | -2, -4           | 22            | 202Gi            |
| gput44       | -2, -4, -8       | 56            | 410Gi            |
| gpuv1004     | -2, -4, -8       | 22            | 422Gi            |


## GPU Time slicing (Short for TS)

Allows up to 8 workloads onto one GPU node to run everything in concurrency, occupying one Node, distributing execution time evenly every ~3 seconds. This way, once auto-scaled, GPU node is available to serve new workloads almost instantly, or, host multiple deployments at the same time.

To utilise time slicing, append 2, 4 or 8 in the compute type: 

Example: __gput44-2__

Another example, if you wish to share A100 node between 4 workloads, specify this: __gpuv1002-4__
This way, it will auto select right Node type to be shared for 4 different workloads: jobs/pipelines/deploys equally distributing resources.

For regular, non time slicing cases, simply indicate compute name without and suffix, e.g.: __gpua1001__



## Specify tags, compute, optional

In [3]:
environment                 = "azureml://registries/azureml/environments/sklearn-1.5/versions/26"
triton_environment          = "tritonNcdEnv:23.08.02-py3"   # Triton Inference Server environment in this workspace
# === Optional: Deployment (MODEL SERVING) details ===
compute_target              = "cpu-2"   # 4 means time slicing into 4 parts on a single node

tags = {
    "Purpose":      "Project Resources",
    "by_person":    "by_person",                        #Highly recommended to specify person name, who submitted the code
}


## Import libraries

In [4]:
import os
from azure.ai.ml import MLClient, command, Input
from azure.identity import ManagedIdentityCredential, DefaultAzureCredential
from azure.ai.ml.entities import (
    CommandComponent,
    PipelineJob,
    JobResourceConfiguration,
    Model,
    KubernetesOnlineEndpoint,
    KubernetesOnlineDeployment,
    Environment,
    CodeConfiguration
)
from azure.ai.ml.dsl import pipeline
from azure.ai.ml.entities._deployment.resource_requirements_settings import (
    ResourceRequirementsSettings,
)
from azure.ai.ml.entities._deployment.container_resource_settings import (
    ResourceSettings,
)
# Service functions
instanceType           = "defaultinstancetype"
nb_prefix              = os.getenv("NB_PREFIX", "/notebook/unknown/unknown")
_, namespace, pod_name = nb_prefix.strip("/").split("/")
AZURE_CLIENT_ID        = os.getenv('AZURE_CLIENT_ID')
client_id              = os.getenv('AZURE_CLIENT_ID')
credential             = ManagedIdentityCredential(client_id=os.getenv("AZURE_CLIENT_ID"))
# === Optional Deployment (MODEL SERVING) details ===
request_cpu = "500m"
request_ram = "2Gi"
limit_cpu = "1"
limit_ram = "4Gi"


## Import Helper script:

In [5]:
from src._helpers.load_tags import load_tags
tags = load_tags(tags={}, namespace=namespace, pod_name=pod_name)
for k, v in tags.items():
    globals()[k] = v

print(tags)

{'CostAllocationCode': 'C.ITD.07.017', 'CostAllocationType': 'WBS', 'Project': 'unified', 'SubProject': 'unified', 'WBS': 'C.ITD.07.017', 'aks_cluster': 'dev-aurora-test-01', 'aml_location': 'northeurope', 'aml_workspace': 'unified-amlws-dev', 'aml_workspace_rg': 'unified-rg-dev', 'subscription_id': '<subscription-id>', 'submitted_by': 'unified', 'notebook_pod': 'test-pshah-medium'}


## Build ML client, specify compute

In [6]:
#  /AML Details 
subscription_id             = subscription_id
resource_group              = aml_workspace_rg
workspace                   = aml_workspace
computeTarget               = f"{compute_target}"
# AML Details/ 

ml_client = MLClient(credential, subscription_id, resource_group, workspace)

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


### Define a 3-stage Iris classification pipeline:

In [7]:
# Pipeline scripts live in ./src/iris_pipeline:
#   train.py    — trains RandomForestClassifier on the Iris dataset, saves model.pkl
#   analysis.py — evaluates the model and writes a classification report
#   score.py    — runs batch inference on sample Iris rows
code_path = "./src/iris_pipeline"


# DEFINE PIPELINE COMPONENTS:
# === 1 ===
train_component = CommandComponent(
    name='iris_train_component',
    display_name='iris_model_training',
    command='python train.py --output_dir ${{outputs.model_output}}',
    code=code_path,
    environment=environment,
    resources=JobResourceConfiguration(instance_count=1, instance_type=instanceType),
    inputs={},
    outputs={'model_output': {'type': 'uri_folder', 'mode': 'rw_mount'}}
)


# === 2 ===
analysis_component = CommandComponent(
    name='iris_analysis_component',
    display_name='iris_model_analysis',
    command='python analysis.py --model_path ${{inputs.model_input}} --output_dir ${{outputs.analysis_output}}',
    code=code_path,
    environment=environment,
    resources=JobResourceConfiguration(instance_count=1, instance_type=instanceType),
    inputs={'model_input': {'type': 'uri_folder'}},
    outputs={'analysis_output': {'type': 'uri_folder', 'mode': 'rw_mount'}}
)


# === 3 ===
inference_component = CommandComponent(
    name='iris_inference_component',
    display_name='iris_inferencing',
    command='python score.py --model_path ${{inputs.model_input}} --output_dir ${{outputs.score_output}}',
    code=code_path,
    environment=environment,
    resources=JobResourceConfiguration(instance_count=1, instance_type=instanceType),
    inputs={'model_input': {'type': 'uri_folder'}},
    outputs={'score_output': {'type': 'uri_folder', 'mode': 'rw_mount'}}
)


# Define the pipeline
@pipeline(default_compute=computeTarget)
def iris_classification_pipeline():
    train_step     = train_component()
    analysis_step  = analysis_component(model_input=train_step.outputs.model_output)
    inference_step = inference_component(model_input=train_step.outputs.model_output)

    return {
        'train_output':    train_step.outputs.model_output,
        'analysis_output': analysis_step.outputs.analysis_output,
        'inference_output': inference_step.outputs.score_output
    }


### Submit Pipeline run:

In [8]:
import threading

pipeline_job = iris_classification_pipeline()

# Add dynamic tags
pipeline_job.tags = tags

print(pipeline_job.tags)
pipeline_job.settings.force_rerun = True
pipeline_job = ml_client.jobs.create_or_update(pipeline_job)

# Stream logs with a hard timeout.
# ml_client.jobs.stream() blocks until the job finishes but has no timeout
# parameter — wrap it in a daemon thread so we can enforce one.
_PIPELINE_TIMEOUT_SEC = 7200  # 2 hours
_stream_exc = []

def _stream_target():
    try:
        ml_client.jobs.stream(pipeline_job.name)
    except Exception as e:
        _stream_exc.append(e)

_stream_thread = threading.Thread(target=_stream_target, daemon=True)
_stream_thread.start()
_stream_thread.join(timeout=_PIPELINE_TIMEOUT_SEC)

if _stream_thread.is_alive():
    raise TimeoutError(
        f"Pipeline job '{pipeline_job.name}' did not finish within "
        f"{_PIPELINE_TIMEOUT_SEC // 3600}h — check AML Studio for status."
    )
if _stream_exc:
    raise _stream_exc[0]

pipeline_run_id = pipeline_job.id
print(pipeline_job.experiment_name)
print(f"Pipeline run ID: {pipeline_run_id}")


Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


{'CostAllocationCode': 'C.ITD.07.017', 'CostAllocationType': 'WBS', 'Project': 'unified', 'SubProject': 'unified', 'WBS': 'C.ITD.07.017', 'aks_cluster': 'dev-aurora-test-01', 'aml_location': 'northeurope', 'aml_workspace': 'unified-amlws-dev', 'aml_workspace_rg': 'unified-rg-dev', 'subscription_id': '<subscription-id>', 'submitted_by': 'unified', 'notebook_pod': 'test-pshah-medium'}


Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


...

RunId: loving_lobster_x1529hgtbv
Web View: https://ml.azure.com/runs/loving_lobster_x1529hgtbv?wsid=/subscriptions/<subscription-id>/resourcegroups/unified-rg-dev/workspaces/unified-amlws-dev



Streaming logs/azureml/executionlogs.txt

[2026-03-03 20:21:16Z] Submitting 1 runs, first five are: ffbc393a:<subscription-id>


[2026-03-03 20:23:26Z] Completing processing run id <subscription-id>.
[2026-03-03 20:23:27Z] Submitting 2 runs, first five are: 318a089b:<subscription-id>,c8132b43:<subscription-id>


[2026-03-03 20:25:35Z] Completing processing run id <subscription-id>.
[2026-03-03 20:25:35Z] Completing processing run id <subscription-id>.

Execution Summary
RunId: loving_lobster_x1529hgtbv
Web View: https://ml.azure.com/runs/loving_lobster_x1529hgtbv?wsid=/subscriptions/<subscription-id>/resourcegroups/unified-rg-dev/workspaces/unified-amlws-dev



ai-platform-aml-triton-examples
Pipeline run ID: /subscriptions/<subscription-id>/resourceGroups/unified-rg-dev/providers/Microsoft.MachineLearningServices/workspaces/unified-amlws-dev/jobs/loving_lobster_x1529hgtbv


### Convert Trained Sklearn Model to ONNX and Build Triton Model Repository

Triton Inference Server's **ONNX Runtime backend** (`backend: "onnxruntime"`) is included in every standard `nvcr/nvidia/tritonserver` image and runs natively on CPU — no GPU required.

> **Why not FIL?**  The FIL (Forest Inference Library) backend is **not compiled into the standard Triton image**. It is only available in separate RAPIDS container builds. Since the workspace environment is based on `tritonserver:23.08-py3`, we use the ONNX Runtime backend instead, which achieves the same result.

The cells below:
1. Download the `model.pkl` (`RandomForestClassifier` on Iris) from the training pipeline child run
2. Convert it to ONNX format with `skl2onnx` using `zipmap=False` so that the probability output is a plain float32 array
3. Assemble the Triton model repository directory structure
4. Write `config.pbtxt` (ONNX Runtime backend, `KIND_CPU`, 4-feature Iris input)

The resulting directory is then registered in Azure ML as a `triton_model` and deployed to the Kubernetes Online Endpoint on the `cpu` compute pool.

**Triton model repository layout:**
```
triton_model_repo/
  iris_classifier/
    config.pbtxt
    1/
      model.onnx
```


In [9]:
import glob as glob_mod
import shutil

# pipeline_job.name is the short job name used by the SDK
pipeline_job_name = pipeline_job.name
# Azure ML sets the child job display_name to the step variable name used inside
# the @pipeline function — here that is "train_step".
training_step_name = "train_step"

# Find the child run that corresponds to the training step
training_run_id = None
child_runs = ml_client.jobs.list(parent_job_name=pipeline_job_name)
for child_run in child_runs:
    print(f"  child: display_name={child_run.display_name!r}  name={child_run.name}")
    if child_run.display_name == training_step_name:
        training_run_id = child_run.name
        print(f"Found training run ID: {training_run_id}")
        break

if training_run_id is None:
    raise RuntimeError(
        f"Could not find a child run named '{training_step_name}' "
        f"under pipeline job '{pipeline_job_name}'. "
        "Check the display_name values printed above and update training_step_name if needed."
    )

# Download the model artifact from the training child run
download_dir = os.path.join(_project_root, "artifacts", "model_download")
os.makedirs(download_dir, exist_ok=True)

ml_client.jobs.download(
    name=training_run_id,
    download_path=download_dir,
    output_name="model_output"
)

# Locate model.pkl — accommodates different Azure ML download path layouts
pkl_candidates = glob_mod.glob(f"{download_dir}/**/model.pkl", recursive=True)
if not pkl_candidates:
    all_files = glob_mod.glob(f"{download_dir}/**/*", recursive=True)
    raise FileNotFoundError(
        f"model.pkl not found under {download_dir}.\n"
        f"Files found: {all_files}\n"
        "Update the filename pattern if your training script uses a different name."
    )

model_pkl_path = pkl_candidates[0]
print(f"Model downloaded to: {model_pkl_path}")


  child: display_name='train_step'  name=<subscription-id>
Found training run ID: <subscription-id>


Model downloaded to: /home/jovyan/ai-platform-aml-triton-examples/artifacts/model_download/named-outputs/model_output/model.pkl


In [10]:
import joblib
import numpy as np
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
from sklearn.ensemble import RandomForestClassifier

# Load the RandomForestClassifier produced by the training pipeline step
model = joblib.load(model_pkl_path)
print(f"Model type      : {type(model)}")
print(f"n_estimators    : {model.n_estimators}")
print(f"n_features_in_  : {model.n_features_in_}")
print(f"classes_        : {model.classes_}")

# Convert to ONNX.
# zipmap=False → probabilities output is a plain float32 array.
# opset=12: broad compatibility with ONNX Runtime 1.15/1.16 inside Triton 23.08.
n_features  = model.n_features_in_    # 4 for Iris
n_classes   = len(model.classes_)     # 3 for Iris
initial_type = [("float_input", FloatTensorType([None, n_features]))]
onnx_model   = convert_sklearn(
    model,
    initial_types=initial_type,
    target_opset=12,
    options={RandomForestClassifier: {"zipmap": False}},
)
print(f"ONNX opset      : {onnx_model.opset_import[0].version}")

# Quick local sanity-check with onnxruntime
import onnxruntime as rt
sess   = rt.InferenceSession(onnx_model.SerializeToString())
sample = np.array([[5.1, 3.5, 1.4, 0.2]], dtype=np.float32)   # setosa
label, proba = sess.run(["label", "probabilities"], {"float_input": sample})
print(f"Local ONNX test : label={label}  proba={proba.round(3)}")

# ── Build the Triton model repository structure ───────────────────────────────
#
#   triton_model_repo/
#     iris_classifier/
#       config.pbtxt
#       1/
#         model.onnx    ← served by Triton ONNX Runtime backend (GPU nodes)
#         model.pkl     ← served by score_ort.py via sklearn  (CPU nodes)
#
import shutil

triton_model_name = "iris_classifier"
triton_model_repo = os.path.join(_project_root, "artifacts", "triton_model_repo")

# Clean up any artefacts from a previous run
if os.path.exists(triton_model_repo):
    shutil.rmtree(triton_model_repo)

model_version_dir = os.path.join(triton_model_repo, triton_model_name, "1")
os.makedirs(model_version_dir, exist_ok=True)

# Save ONNX model (for Triton serving on GPU nodes)
onnx_path = os.path.join(model_version_dir, "model.onnx")
with open(onnx_path, "wb") as f:
    f.write(onnx_model.SerializeToString())
print(f"ONNX model saved : {onnx_path}")

# Also copy the sklearn pickle (for score_ort.py serving on CPU nodes)
pkl_dest = os.path.join(model_version_dir, "model.pkl")
shutil.copy2(model_pkl_path, pkl_dest)
print(f"sklearn pkl saved: {pkl_dest}")


/home/jovyan/triton-env/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.5.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/jovyan/triton-env/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.5.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Model type      : <class 'sklearn.ensemble._forest.RandomForestClassifier'>
n_estimators    : 10
n_features_in_  : 4
classes_        : [0 1 2]
ONNX opset      : 1


Local ONNX test : label=[0]  proba=[[1. 0. 0.]]
ONNX model saved : /home/jovyan/ai-platform-aml-triton-examples/artifacts/triton_model_repo/iris_classifier/1/model.onnx
sklearn pkl saved: /home/jovyan/ai-platform-aml-triton-examples/artifacts/triton_model_repo/iris_classifier/1/model.pkl


In [11]:
# Create Triton ONNX Runtime backend configuration file (config.pbtxt).
#
# ONNX Runtime backend is always present in nvcr/nvidia/tritonserver images
# and supports CPU inference natively via KIND_CPU.
#
# Input/output names must match exactly what skl2onnx exported:
#   input  → "float_input"  (FloatTensorType, 4 Iris features)
#   output → "label"        (INT64, predicted class 0/1/2)
#   output → "probabilities" (FP32, [3] per sample — exposed but optional to query)

config_content = f"""name: "{triton_model_name}"
backend: "onnxruntime"
max_batch_size: 64

input [
  {{
    name: "float_input"
    data_type: TYPE_FP32
    dims: [{n_features}]
  }}
]

output [
  {{
    name: "label"
    data_type: TYPE_INT64
    dims: [1]
  }},
  {{
    name: "probabilities"
    data_type: TYPE_FP32
    dims: [3]
  }}
]

instance_group [
  {{
    kind: KIND_CPU
    count: 1
  }}
]

dynamic_batching {{
  max_queue_delay_microseconds: 100
}}
"""

config_path = os.path.join(triton_model_repo, triton_model_name, "config.pbtxt")
with open(config_path, "w") as f:
    f.write(config_content)

print("Triton ONNX Runtime config.pbtxt created:")
print(config_content)
print(f"Triton model repository ready at: {triton_model_repo}")


Triton ONNX Runtime config.pbtxt created:
name: "iris_classifier"
backend: "onnxruntime"
max_batch_size: 64

input [
  {
    name: "float_input"
    data_type: TYPE_FP32
    dims: [4]
  }
]

output [
  {
    name: "label"
    data_type: TYPE_INT64
    dims: [1]
  },
  {
    name: "probabilities"
    data_type: TYPE_FP32
    dims: [3]
  }
]

instance_group [
  {
    kind: KIND_CPU
    count: 1
  }
]

dynamic_batching {
  max_queue_delay_microseconds: 100
}

Triton model repository ready at: /home/jovyan/ai-platform-aml-triton-examples/artifacts/triton_model_repo


## Train PyTorch Sine-Wave Model Locally

Train a small 3-layer MLP to approximate `y = sin(x)` for `x ∈ [−π, π]`.

**Architecture:** `Linear(1→32) → ReLU → Linear(32→32) → ReLU → Linear(32→1)`

Training is done locally in the notebook (fast, no AML job needed).
The model is then:
1. Exported to **ONNX** — for serving by Triton ONNX Runtime backend on GPU nodes
2. Saved as **numpy `.npz` weights** — for CPU serving via pure-numpy forward pass (no CUDA or onnxruntime dependency at serving time)

Triton model repository entry:
```
triton_model_repo/
  pytorch_sine/
    config.pbtxt   (backend: onnxruntime, input: x[1], output: y[1])
    1/
      model.onnx        ← GPU / native Triton
      model_params.npz  ← CPU / score_multi_ort.py numpy serving
```

In [12]:
import torch
import torch.nn as nn
import numpy as np

# ── Toy MLP architecture ────────────────────────────────────────────────────
class SineMLP(nn.Module):
    """3-layer MLP that approximates y = sin(x)."""
    def __init__(self, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x)

# ── Generate training data ───────────────────────────────────────────────────
torch.manual_seed(42)
np.random.seed(42)
N = 1000
X_pt = torch.linspace(-np.pi, np.pi, N).unsqueeze(1)  # [N, 1]
y_pt = torch.sin(X_pt) + 0.05 * torch.randn_like(X_pt)

# ── Train ────────────────────────────────────────────────────────────────────
model_pt = SineMLP(hidden=32)
optimizer = torch.optim.Adam(model_pt.parameters(), lr=1e-3)
loss_fn   = nn.MSELoss()

for epoch in range(3000):
    optimizer.zero_grad()
    loss = loss_fn(model_pt(X_pt), y_pt)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 500 == 0:
        print(f"Epoch {epoch+1:4d}  loss={loss.item():.6f}")

model_pt.eval()
print(f"\nTraining complete — final MSE loss: {loss.item():.6f}")

# ── Quick local sanity check ─────────────────────────────────────────────────
with torch.no_grad():
    test_xs = [0.0, float(np.pi / 2), float(np.pi)]
    for xi in test_xs:
        yi = model_pt(torch.tensor([[xi]])).item()
        print(f"  sin({xi:+.4f}) ≈ {yi:+.4f}  (true: {np.sin(xi):+.4f})")


Epoch  500  loss=0.002641


Epoch 1000  loss=0.002496


Epoch 1500  loss=0.002472


Epoch 2000  loss=0.002457


Epoch 2500  loss=0.002448


Epoch 3000  loss=0.002444

Training complete — final MSE loss: 0.002444
  sin(+0.0000) ≈ -0.0032  (true: +0.0000)
  sin(+1.5708) ≈ +0.9988  (true: +1.0000)
  sin(+3.1416) ≈ -0.0020  (true: +0.0000)


In [13]:
import torch.onnx
import shutil

pytorch_model_name = "pytorch_sine"
pytorch_version_dir = os.path.join(triton_model_repo, pytorch_model_name, "1")
os.makedirs(pytorch_version_dir, exist_ok=True)

# ── Export to ONNX (for GPU nodes with native Triton) ────────────────────────
dummy_input = torch.tensor([[0.0]])
onnx_pt_path = os.path.join(pytorch_version_dir, "model.onnx")
torch.onnx.export(
    model_pt,
    dummy_input,
    onnx_pt_path,
    input_names=["x"],
    output_names=["y"],
    dynamic_axes={"x": {0: "batch_size"}, "y": {0: "batch_size"}},
    opset_version=12,
    dynamo=False,   # use legacy TorchScript exporter (no onnxscript dependency)
)
print(f"ONNX model saved  : {onnx_pt_path}")

# ── Local ONNX sanity check ──────────────────────────────────────────────────
import onnxruntime as rt
sess_pt = rt.InferenceSession(onnx_pt_path)
ort_x   = np.array([[np.pi / 2]], dtype=np.float32)
ort_out = sess_pt.run(["y"], {"x": ort_x})[0]
print(f"ONNX check: sin(π/2) ≈ {ort_out[0,0]:.4f}  (true: {np.sin(np.pi/2):.4f})")

# ── Save numpy weights for CPU serving ───────────────────────────────────────
# score_multi_ort.py runs a pure-numpy forward pass at inference time,
# so the serving container needs no onnxruntime or CUDA dependency.
linears = [m for m in model_pt.net if isinstance(m, nn.Linear)]
params_dict = {}
for idx, layer in enumerate(linears, start=1):
    params_dict[f"W{idx}"] = layer.weight.detach().numpy()
    params_dict[f"b{idx}"] = layer.bias.detach().numpy()

npz_path = os.path.join(pytorch_version_dir, "model_params.npz")
np.savez(npz_path, **params_dict)
print(f"Numpy weights saved: {npz_path}")

# ── Numpy forward-pass sanity check ─────────────────────────────────────────
def _relu_np(x): return np.maximum(0.0, x)
test_x_np = np.array([[np.pi / 2]], dtype=np.float32)
h = _relu_np(test_x_np @ params_dict["W1"].T + params_dict["b1"])
h = _relu_np(h         @ params_dict["W2"].T + params_dict["b2"])
y_np = h @ params_dict["W3"].T + params_dict["b3"]
print(f"Numpy  check: sin(π/2) ≈ {y_np[0,0]:.4f}  (true: {np.sin(np.pi/2):.4f})")


/tmp/ipykernel_77273/2685843583.py:11: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX model saved  : /home/jovyan/ai-platform-aml-triton-examples/artifacts/triton_model_repo/pytorch_sine/1/model.onnx
ONNX check: sin(π/2) ≈ 0.9988  (true: 1.0000)
Numpy weights saved: /home/jovyan/ai-platform-aml-triton-examples/artifacts/triton_model_repo/pytorch_sine/1/model_params.npz
Numpy  check: sin(π/2) ≈ 0.9988  (true: 1.0000)


In [14]:
# ── Triton config.pbtxt for pytorch_sine ────────────────────────────────────
# ONNX Runtime backend; input tensor "x" (FP32 [1]), output "y" (FP32 [1]).
# max_batch_size 64 matches iris_classifier for consistency.
pytorch_config = f"""name: "{pytorch_model_name}"
backend: "onnxruntime"
max_batch_size: 64

input [
  {{
    name: "x"
    data_type: TYPE_FP32
    dims: [1]
  }}
]

output [
  {{
    name: "y"
    data_type: TYPE_FP32
    dims: [1]
  }}
]

instance_group [
  {{
    kind: KIND_CPU
    count: 1
  }}
]

dynamic_batching {{
  max_queue_delay_microseconds: 100
}}
"""

pytorch_config_path = os.path.join(triton_model_repo, pytorch_model_name, "config.pbtxt")
with open(pytorch_config_path, "w") as f:
    f.write(pytorch_config)
print("pytorch_sine config.pbtxt written.")

# ── Print final Triton model repository layout ───────────────────────────────
print(f"\nTriton model repository layout: {triton_model_repo}")
for root, dirs, files in os.walk(triton_model_repo):
    level = root.replace(triton_model_repo, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for fname in sorted(files):
        fpath = os.path.join(root, fname)
        print(f"{indent}  {fname}  ({os.path.getsize(fpath):,} bytes)")


pytorch_sine config.pbtxt written.

Triton model repository layout: /home/jovyan/ai-platform-aml-triton-examples/artifacts/triton_model_repo
triton_model_repo/
  pytorch_sine/
    config.pbtxt  (326 bytes)
    1/
      model.onnx  (5,416 bytes)
      model_params.npz  (6,050 bytes)
  iris_classifier/
    config.pbtxt  (418 bytes)
    1/
      model.onnx  (8,270 bytes)
      model.pkl  (19,425 bytes)


### Register Triton Multi-Model Repository in Azure ML

In [15]:
# Register the Triton FIL model repository as an Azure ML model (type: triton_model).
# Azure ML uploads the local directory structure to blob storage and tracks it as a versioned asset.
import datetime

model_name        = "triton-iris-fil-" + datetime.datetime.now().strftime("%Y%m%d%H%M%S")
model_description = (
    "Two-model Triton repository: iris_classifier (sklearn RFC) + "
    "pytorch_sine (MLP sine regressor). KFServing V2 API. CPU inference."
)

model = Model(
    name=model_name,
    path=triton_model_repo,   # local Triton model repository directory — Azure ML uploads it
    type="triton_model",       # tells Azure ML this is a Triton model repository
    description=model_description
)

registered_model = ml_client.models.create_or_update(model)
print(f"Triton FIL model registered : {registered_model.name}")
print(f"Model ID                    : {registered_model.id}")


Uploading triton_model_repo (0.04 MBs):   0%|          | 0/39905 [00:00<?, ?it/s]

Uploading triton_model_repo (0.04 MBs):  16%|█▌        | 6468/39905 [00:00<00:00, 64392.97it/s]

Uploading triton_model_repo (0.04 MBs): 100%|██████████| 39905/39905 [00:00<00:00, 322408.20it/s]

Triton FIL model registered : triton-iris-fil-20260303202754
Model ID                    : /subscriptions/<subscription-id>/resourceGroups/unified-rg-dev/providers/Microsoft.MachineLearningServices/workspaces/unified-amlws-dev/models/triton-iris-fil-20260303202754/versions/1


## 2. Deploy online endpoint to Azure


### 2.1 Configure online endpoint
`endpoint_name`: The name of the endpoint.

`auth_mode` : Use `key` for key-based authentication. Use `aml_token` for Azure Machine Learning token-based authentication. A `key` does not expire, but `aml_token` does expire. 

Optionally, you can add description, tags to your endpoint.

In [16]:
# ── Cleanup: Delete orphaned endpoints to free RBAC role assignment quota ─────
# Azure subscriptions have a limit on role assignments (typically 2000 per scope).
# Each endpoint deployment creates role assignments. Orphaned endpoints (no traffic)
# consume quota without serving anything. This cell deletes old ai-platform-endpoint*
# endpoints that have no active deployments to free up quota before creating a new one.
#
# To skip cleanup and reuse an existing endpoint, comment out this cell and set:
#   online_endpoint_name = "ai-platform-endpoint<timestamp>"  # existing name

ENDPOINT_PREFIX = "ai-platform-endpoint"

try:
    existing_endpoints = list(ml_client.online_endpoints.list())
    orphans = [
        ep for ep in existing_endpoints
        if ep.name.startswith(ENDPOINT_PREFIX) and not ep.traffic
    ]
    if orphans:
        print(f"Found {len(orphans)} orphaned endpoint(s) to clean up:")
        for ep in orphans:
            print(f"  Deleting: {ep.name}")
            try:
                ml_client.online_endpoints.begin_delete(name=ep.name).result()
                print(f"  Deleted: {ep.name}")
            except Exception as del_err:
                print(f"  Warning: could not delete {ep.name}: {del_err}")
    else:
        print("No orphaned endpoints found — quota should be available.")
except Exception as e:
    print(f"Warning: endpoint cleanup check failed: {e}")
    print("Continuing with endpoint creation...")


No orphaned endpoints found — quota should be available.


In [17]:
# deploy the model to AKS
import datetime


online_endpoint_name= "ai-platform-endpoint"+ datetime.datetime.now().strftime("%m%d%f")
print(f"Online Endpoint name is: {online_endpoint_name}")

# create an online endpoint
endpoint= KubernetesOnlineEndpoint(
    name=online_endpoint_name,
    compute=computeTarget,
    description="AI Plaatforms endpoint with SSL Termination",
    auth_mode="key",
    tags=tags,
)

_ENDPOINT_TIMEOUT_SEC = 600  # 10 min
ml_client.begin_create_or_update(endpoint).result(timeout=_ENDPOINT_TIMEOUT_SEC)

Online Endpoint name is: ai-platform-endpoint0303358143


KubernetesOnlineEndpoint({'provisioning_state': 'Succeeded', 'scoring_uri': 'https://unified.aurora.equinor.com/api/v1/endpoint/ai-platform-endpoint0303358143/score', 'openapi_uri': 'https://unified.aurora.equinor.com/api/v1/endpoint/ai-platform-endpoint0303358143/swagger.json', 'name': 'ai-platform-endpoint0303358143', 'description': 'AI Plaatforms endpoint with SSL Termination', 'tags': {'CostAllocationCode': 'C.ITD.07.017', 'CostAllocationType': 'WBS', 'Project': 'unified', 'SubProject': 'unified', 'WBS': 'C.ITD.07.017', 'aks_cluster': 'dev-aurora-test-01', 'aml_location': 'northeurope', 'aml_workspace': 'unified-amlws-dev', 'aml_workspace_rg': 'unified-rg-dev', 'subscription_id': '<subscription-id>', 'submitted_by': 'unified', 'notebook_pod': 'test-pshah-medium', 'ApprovedBy': 'OANA@equinor.com', 'CostOwner': 'JRIVE@equinor.com', 'MainContact': 'MYP@equinor.com', 'Purpose': 'Project Resources', 'SecondaryContact': 'PSHAH@equinor.com'}, 'properties': {'createdBy': '<subscription-id>

## Configure Multi-Model Deployment

`score_multi_ort.py` serves **both models** from a single deployment using the **rawhttp** decorator from `azureml_inference_server_http`. This gives full control over the HTTP request/response, enabling Triton-style per-model URL routing.

**Routing priority** (scoring script tries each in order):

| Priority | Mechanism | Example URL |
|----------|-----------|-------------|
| 1 | Triton path | `POST {uri}/v2/models/iris_classifier/infer` |
| 2 | Query param | `POST {uri}?model=pytorch_sine` |
| 3 | Body field  | `{"model_name": "iris_classifier", "inputs": [...]}` |

**Model control mode** — V2 management endpoints also available:
- `GET  {uri}/v2/health/ready` — server health
- `GET  {uri}/v2/models/{name}/ready` — per-model readiness
- `GET  {uri}/v2/repository/index` — list loaded models
- `POST {uri}/v2/repository/models/{name}/load` — hot-load a model
- `POST {uri}/v2/repository/models/{name}/unload` — unload a model

> **Why not native Triton?** `tritonserver` images require NVIDIA GPU drivers. `score_multi_ort.py` uses sklearn + pure-numpy — zero CUDA dependency.

In [18]:
# Parameters for CPU deployment.
#
# serving_environment  — sklearn-1.5 curated env (has joblib/numpy/sklearn, no CUDA needed)
# triton_scoring_path  — ./src/triton_scoring  (shared with Triton proxy; score_ort.py lives here)
# triton_scoring_script — score_ort.py implements KFServing V2 API via sklearn inference
#
# To switch to native Triton on a GPU node:
#   serving_environment  = triton_environment
#   triton_scoring_script = "score.py"
serving_environment   = environment          # sklearn-1.5: proven to work on CPU AKS nodes
deployment_name       = "blue-deployment"
endpoint_name         = online_endpoint_name
model_id              = registered_model.id
instance_count        = 1
triton_scoring_path   = "./src/triton_scoring"
triton_scoring_script = "score_multi_ort.py"  # Multi-model rawhttp V2 (CPU-safe)


### Create Deployment object 

In [19]:
# CPU deployment using sklearn-1.5 environment + KFServing V2 scoring script.
# score_multi_ort.py uses rawhttp mode to route requests by model name.
# Serves both iris_classifier and pytorch_sine from a single deployment.
blue_deployment = KubernetesOnlineDeployment(
    name=deployment_name,
    endpoint_name=endpoint_name,
    model=model_id,
    environment=serving_environment,          # sklearn-1.5 (no CUDA)
    code_configuration=CodeConfiguration(
        code=triton_scoring_path,
        scoring_script=triton_scoring_script,  # score_ort.py
    ),
    instance_count=instance_count,
    instance_type=instanceType,
    tags=tags,
)


## 3.1 Create the deployment and set traffic

Using the `MLClient` created earlier, we will now create the deployment in the workspace. This command will start the deployment creation and return a confirmation response while the deployment creation continues.

In [20]:
_DEPLOY_TIMEOUT_SEC  = 1800  # 30 min — image pull + model init
_TRAFFIC_TIMEOUT_SEC = 300   # 5 min  — traffic routing update only
ml_client.begin_create_or_update(blue_deployment).result(timeout=_DEPLOY_TIMEOUT_SEC)

endpoint = ml_client.online_endpoints.get(name=endpoint_name)
scoring_uri = endpoint.scoring_uri
print("Scoring URI:", scoring_uri)
endpoint.traffic = {"blue-deployment": 100}
ml_client.begin_create_or_update(endpoint).result(timeout=_TRAFFIC_TIMEOUT_SEC)

# ── Wait for the scoring script to warm up ────────────────────────────────────
# After deployment, the container and gunicorn workers need time to start.
# Poll until the endpoint accepts requests (HTTP 200 or 4xx), not 502/503.
import time, requests as _req
print()
print("Waiting for endpoint to be ready (may take 1-3 min)...")
endpoint_keys = ml_client.online_endpoints.get_keys(name=endpoint_name)
_test_headers = {
    "Authorization": f"Bearer {endpoint_keys.primary_key}",
    "Content-Type": "application/json",
}
_test_payload = {
    "inputs": [{"name": "float_input", "shape": [1, 4],
                "datatype": "FP32", "data": [[5.1, 3.5, 1.4, 0.2]]}],
    "outputs": [{"name": "label"}],
}
for _attempt in range(24):   # up to 6 minutes (24 x 15 s)
    try:
        _r = _req.post(
            f"{scoring_uri}?model=iris_classifier",
            headers=_test_headers, json=_test_payload, verify=False, timeout=15
        )
        print(f"  attempt {_attempt+1:2d}: HTTP {_r.status_code}")
        if _r.status_code not in (502, 503, 504):
            print("Endpoint is ready.")
            break
    except Exception as _e:
        print(f"  attempt {_attempt+1:2d}: connection error — {_e}")
    time.sleep(15)
else:
    print("WARNING: endpoint did not become ready within 6 minutes.")

print()
print("Check: endpoint", endpoint_name, "exists")


Check: endpoint ai-platform-endpoint0303358143 exists


Uploading triton_scoring (0.04 MBs):   0%|          | 0/35424 [00:00<?, ?it/s]

Uploading triton_scoring (0.04 MBs):  35%|███▍      | 12331/35424 [00:00<00:00, 109410.69it/s]

Uploading triton_scoring (0.04 MBs): 100%|██████████| 35424/35424 [00:00<00:00, 129269.27it/s]

Uploading triton_scoring (0.04 MBs): 100%|██████████| 35424/35424 [00:00<00:00, 126294.92it/s]

...

Readonly attribute principal_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>


Readonly attribute tenant_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>



Waiting for endpoint to be ready (may take 1-3 min)...


  attempt  1: HTTP 200
Endpoint is ready.

Check: endpoint ai-platform-endpoint0303358143 exists


/home/jovyan/triton-env/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'unified.aurora.equinor.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


## Test Multi-Model Endpoint

Each model has a **distinct URL** for inference via **query-parameter routing**:

| Model | Working URL (query-param) |
|-------|---------------------------|
| `iris_classifier` | `{uri}?model=iris_classifier` |
| `pytorch_sine`    | `{uri}?model=pytorch_sine` |

> **AML Gateway Routing Note**: The AML Kubernetes gateway only forwards requests to the
>  path. Sub-path routing (e.g. ) returns HTTP 404
> from the gateway before the scoring script is reached. Query-parameter routing
> () is fully forwarded and is the recommended approach.

The V2 management endpoints (health, readiness, repository index) are implemented in the
scoring script but are inaccessible from outside via the AML gateway (same sub-path limitation).
They are available for internal/sidecar use within the container.


In [21]:
import requests
import json

# Retrieve endpoint credentials
endpoint_keys = ml_client.online_endpoints.get_keys(name=endpoint_name)
api_key       = endpoint_keys.primary_key

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type":  "application/json",
}

# ── V2 management endpoints ──────────────────────────────────────────────────
# Note: AML gateway does NOT forward sub-paths beyond /score -> expect HTTP 404.
# These endpoints are implemented in the scoring script for internal/sidecar access.
health_url = f"{scoring_uri}/v2/health/ready"
resp = requests.get(health_url, headers=headers, verify=False)
msg = "OK" if resp.status_code == 200 else "404 (expected - AML gateway blocks sub-paths)"
print(f"Health check [{resp.status_code}]: {msg}")

# ── Test iris_classifier ─────────────────────────────────────────────────────
print()
print("=" * 60)
print("MODEL 1: iris_classifier")
print("=" * 60)

iris_payload = {
    "inputs": [{
        "name":     "float_input",
        "shape":    [1, 4],
        "datatype": "FP32",
        "data":     [[5.1, 3.5, 1.4, 0.2]]   # classic setosa sample
    }],
    "outputs": [{"name": "label"}, {"name": "probabilities"}]
}

# Primary: query-parameter routing (confirmed working with AML gateway)
iris_url = f"{scoring_uri}?model=iris_classifier"
resp = requests.post(iris_url, headers=headers, json=iris_payload, verify=False)
print(f"  [query param] -> HTTP {resp.status_code}  url={iris_url}")

iris_result = None
if resp.status_code == 200:
    raw = resp.text
    try:
        iris_result = json.loads(raw)
        if isinstance(iris_result, str):
            iris_result = json.loads(iris_result)
    except Exception:
        iris_result = {"raw": raw}

if iris_result:
    class_names = ["setosa", "versicolor", "virginica"]
    print()
    print("Response:")
    print(json.dumps(iris_result, indent=2))
    for out in iris_result.get("outputs", []):
        if out["name"] == "label":
            cls = int(out["data"][0])
            print()
            print(f"Predicted class  : {cls} ({class_names[cls]})")
        if out["name"] == "probabilities":
            probs = out["data"]
            print("Class probabilities: " +
                  "  ".join(f"{class_names[k]}={probs[k]:.3f}" for k in range(3)))
else:
    print()
    print(f"ERROR: {resp.status_code} - {resp.text[:300]}")


Health check [404]: 404 (expected - AML gateway blocks sub-paths)

MODEL 1: iris_classifier
  [query param] -> HTTP 200  url=https://unified.aurora.equinor.com/api/v1/endpoint/ai-platform-endpoint0303358143/score?model=iris_classifier

Response:
{
  "model_name": "iris_classifier",
  "model_version": "1",
  "outputs": [
    {
      "name": "label",
      "shape": [
        1
      ],
      "datatype": "INT64",
      "data": [
        0
      ]
    },
    {
      "name": "probabilities",
      "shape": [
        1,
        3
      ],
      "datatype": "FP32",
      "data": [
        1.0,
        0.0,
        0.0
      ]
    }
  ]
}

Predicted class  : 0 (setosa)
Class probabilities: setosa=1.000  versicolor=0.000  virginica=0.000


/home/jovyan/triton-env/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'unified.aurora.equinor.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/jovyan/triton-env/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'unified.aurora.equinor.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


### Test MODEL 2: pytorch_sine

Send three x-values and compare the predicted y ≈ sin(x) with the ground truth.

In [22]:
# ── Test pytorch_sine ────────────────────────────────────────────────────────
print("MODEL 2: pytorch_sine")
print("=" * 60)

import math
test_xs = [0.0, math.pi / 2, math.pi]

sine_payload = {
    "inputs": [{
        "name":     "x",
        "shape":    [3, 1],
        "datatype": "FP32",
        "data":     [[xi] for xi in test_xs]
    }],
    "outputs": [{"name": "y"}]
}

# Primary: query-parameter routing (confirmed working with AML gateway)
sine_url = f"{scoring_uri}?model=pytorch_sine"
resp = requests.post(sine_url, headers=headers, json=sine_payload, verify=False)
print(f"  [query param] -> HTTP {resp.status_code}  url={sine_url}")

sine_result = None
if resp.status_code == 200:
    raw = resp.text
    try:
        sine_result = json.loads(raw)
        if isinstance(sine_result, str):
            sine_result = json.loads(sine_result)
    except Exception:
        sine_result = {"raw": raw}

if sine_result:
    print()
    print("Response:")
    print(json.dumps(sine_result, indent=2))
    for out in sine_result.get("outputs", []):
        if out["name"] == "y":
            ys = out["data"]
            print()
            print("Predictions vs ground truth:")
            for xi, yi in zip(test_xs, ys):
                print(f"  sin({xi:+.4f}) -> predicted={yi:+.4f}  true={math.sin(xi):+.4f}  "
                      f"error={abs(yi - math.sin(xi)):.4f}")
else:
    print()
    print(f"ERROR: {resp.status_code} - {resp.text[:300]}")

# ── Curl equivalents ──────────────────────────────────────────────────────────
print()
print("=" * 60)
print("CURL EQUIVALENTS (query-param routing)")
print("=" * 60)
for model, sample in [
    ("iris_classifier",
     '{"inputs":[{"name":"float_input","shape":[1,4],"datatype":"FP32","data":[[5.1,3.5,1.4,0.2]]}]}'),
    ("pytorch_sine",
     '{"inputs":[{"name":"x","shape":[1,1],"datatype":"FP32","data":[[1.5708]]}]}'),
]:
    curl_url = f"{scoring_uri}?model={model}"
    print()
    print(f"# {model}")
    print(f"curl -k -X POST '{curl_url}' \\")
    print(f"  -H 'Authorization: Bearer $API_KEY' \\")
    print(f"  -H 'Content-Type: application/json' \\")
    print(f"  -d '{sample}'")


MODEL 2: pytorch_sine
  [query param] -> HTTP 200  url=https://unified.aurora.equinor.com/api/v1/endpoint/ai-platform-endpoint0303358143/score?model=pytorch_sine

Response:
{
  "model_name": "pytorch_sine",
  "model_version": "1",
  "outputs": [
    {
      "name": "y",
      "shape": [
        3,
        1
      ],
      "datatype": "FP32",
      "data": [
        -0.0032458752393722534,
        0.9988261461257935,
        -0.0019884854555130005
      ]
    }
  ]
}

Predictions vs ground truth:
  sin(+0.0000) -> predicted=-0.0032  true=+0.0000  error=0.0032
  sin(+1.5708) -> predicted=+0.9988  true=+1.0000  error=0.0012
  sin(+3.1416) -> predicted=-0.0020  true=+0.0000  error=0.0020

CURL EQUIVALENTS (query-param routing)

# iris_classifier
curl -k -X POST 'https://unified.aurora.equinor.com/api/v1/endpoint/ai-platform-endpoint0303358143/score?model=iris_classifier' \
  -H 'Authorization: Bearer $API_KEY' \
  -H 'Content-Type: application/json' \
  -d '{"inputs":[{"name":"float_input"

/home/jovyan/triton-env/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'unified.aurora.equinor.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
